In [ ]:
import os
# os.environ["MUJOCO_GL"] = "glfw"

# dataset
from lerobot.datasets.transforms import ImageTransformsConfig
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

# policy
from lerobot.policies.act.configuration_act import ACTConfig

# lerobot training
from lerobot.configs.train import TrainPipelineConfig, DatasetConfig
from lerobot.optim.optimizers import AdamWConfig
from lerobot.configs.default import WandBConfig
from lerobot.utils.utils import init_logging
from lerobot.scripts import train

# torch
from torch.cuda import is_available

# wandb
import wandb

# utils
from dotenv import load_dotenv
from pprint import pprint
import sys

# my code
from src.paths import REPO_ROOT, DATASETS_DIR, HF_NAME, POLICIES_DIR

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)


C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Must be run as admin!

In [4]:
device = "cuda" if is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [5]:
PUSH_TO_HUB = False
REPO_NAME = 'so101_leg_pick_and_place'
EXPERIMENT_NAME = 'test'
resume_checkpoint = None
bad_eps = {}

In [6]:
DATASET_PATH = DATASETS_DIR / REPO_NAME
DATASET_ID = f"{HF_NAME}/{REPO_NAME}"

In [7]:
if PUSH_TO_HUB:
    HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
    os.sys(f"huggingface-cli login --token {HF_TOKEN}")

In [16]:
wandb.login(key = os.getenv('WANDB_API_KEY'))

wandb: Currently logged in as: jonathm126 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [17]:
ds_meta = LeRobotDatasetMetadata(DATASET_ID, root = DATASET_PATH)
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"\nRobot type: {ds_meta.robot_type}")
print(f"\nkeys to access images from cameras: {ds_meta.camera_keys}")
print("\nFeatures:")
pprint(ds_meta.features.keys(), width = 40 )
print('\nAction Space:')
pprint(ds_meta.features['action'])

Total number of episodes: 50
Average number of frames per episode: 753.300
Frames per second used during data collection: 30.0

Robot type: so101_follower

keys to access images from cameras: ['observation.images.wrist_cam', 'observation.images.top_cam']

Features:
dict_keys(['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'])

Action Space:
{'dtype': 'float32',
 'fps': 30.0,
 'names': ['shoulder_pan.pos',
           'shoulder_lift.pos',
           'elbow_flex.pos',
           'wrist_flex.pos',
           'wrist_roll.pos',
           'gripper.pos'],
 'shape': (6,)}


In [ ]:
# select image transforms
transforms_cfg = ImageTransformsConfig(enable = True,
                                        max_num_transforms = 3,
                                        random_order = True,
                                        )

# select episodes
episodes = [ep for ep in range(len(ds_meta.episodes)) if ep not in bad_eps]

# build dataset
dataset_cfg = DatasetConfig(repo_id            = DATASET_ID,
                            root               = DATASET_PATH,     # dataset location
                            episodes           = episodes,         # which episodes to use
                            image_transforms   = transforms_cfg,   # configure image transformations
                            use_imagenet_stats = True,             # normalize image using imagenet weights
                            video_backend      = 'pyav',           # for windows
                            streaming          = False
                            )

Policy Config

In [19]:
input_features = {
    "observation.images.wrist_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.images.top_cam": {
        "type" : "VISUAL",
        "shape": [3, 480, 640]
    },
    "observation.state": {
        "type" : "STATE",
        "shape": [6]
    }
}

output_features = {
    "action": {
        "type" : "ACTION",
        "shape": [6]
    }
}

In [20]:
policy_config = ACTConfig(n_obs_steps = 1, # how many observations does the policy need
                            chunk_size              = 50,             # action chunking size
                            n_action_steps          = 50,               # how many actions to preform (?) TODO - reduce to 10
                            device                  = device,
                            input_features          = input_features,
                            output_features         = output_features,
                            push_to_hub             = False,
                            temporal_ensemble_coeff = None,
                            use_vae                 = True
                            )

Training params config:

In [21]:
optimizer_cfg = AdamWConfig(lr = 1e-5, weight_decay = 1e-4)

WandB logging:

In [22]:
wandb_config = WandBConfig(enable = True, 
                            disable_artifact = False,
                            project          = 'ACT-Real-Sim-Real',
                            entity           = 'jonathm126-personal',
                            mode             = 'online',
                            run_id           =  EXPERIMENT_NAME + '-train' ) # careful - this is a unique ids
print(wandb_config.run_id)

test-train


Integration:

In [23]:
train_cfg = TrainPipelineConfig(dataset                    = dataset_cfg,
                                # env                        = env_config,
                                policy                     = policy_config,                                                     # policy config
                                output_dir                 = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME,
                                job_name                   = REPO_NAME + '-' + EXPERIMENT_NAME + '-train',                      # name
                                num_workers                = 4,
                                batch_size                 = 2,
                                steps                      = 100000,
                                eval_freq                  = 20000,
                                log_freq                   = 200,
                                save_checkpoint            = True,
                                save_freq                  = 20000,
                                use_policy_training_preset = True,
                                optimizer                  = optimizer_cfg,
                                scheduler                  = None,
                                eval                       = None,
                                wandb                      = wandb_config,
                                )

Train:

In [ ]:
if resume_checkpoint is None:
    init_logging()
    train.train(train_cfg)
else:
    assert resume_checkpoint is not None
    pth = POLICIES_DIR / policy_config.type / REPO_NAME / EXPERIMENT_NAME / 'checkpoints' / resume_checkpoint / 'pretrained_model' / 'train_config.json'
    assert pth.exists()
    sys.argv = [
        "train",
        f"--config_path={pth}",
        "--resume=true",
    ]
    init_logging()
    train.train()

INFO 2025-09-23 15:29:17 ts\train.py:148 {'batch_size': 6,
 'dataset': {'episodes': [0,
                          1,
                          2,
                          3,
                          4,
                          5,
                          6,
                          7,
                          8,
                          9,
                          10,
                          11,
                          12,
                          13,
                          14,
                          15,
                          16,
                          17,
                          18,
                          19,
                          20,
                          21,
                          22,
                          23,
                          24,
                          25,
                          26,
                          27,
                          28,
                          29,
                          30,
                     

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.
INFO 2025-09-23 15:29:19 db_utils.py:103 Track this run --> https://wandb.ai/jonathm126-personal/ACT-Real-Sim-Real/runs/test-train
INFO 2025-09-23 15:29:19 ts\train.py:164 Creating dataset


Logs will be synced with wandb.


INFO 2025-09-23 15:29:19 ts\train.py:175 Creating policy
INFO 2025-09-23 15:29:20 ts\train.py:194 Creating optimizer and scheduler
INFO 2025-09-23 15:29:20 ts\train.py:206 Output dir: C:\Github\IDC_Project-Sim_Real_Sim\policies\act\so101_leg_pick_and_place\test
INFO 2025-09-23 15:29:20 ts\train.py:209 cfg.steps=100000 (100K)
INFO 2025-09-23 15:29:20 ts\train.py:210 dataset.num_frames=34057 (34K)
INFO 2025-09-23 15:29:20 ts\train.py:211 dataset.num_episodes=50
INFO 2025-09-23 15:29:20 ts\train.py:212 num_learnable_params=51571590 (52M)
INFO 2025-09-23 15:29:20 ts\train.py:213 num_total_params=51571590 (52M)
INFO 2025-09-23 15:29:20 ts\train.py:254 Start offline training on a fixed dataset
INFO 2025-09-23 15:30:27 ts\train.py:281 step:200 smpl:1K ep:2 epch:0.04 loss:7.429 grdn:184.670 lr:1.0e-05 updt_s:0.305 data_s:0.028
INFO 2025-09-23 15:31:27 ts\train.py:281 step:400 smpl:2K ep:4 epch:0.07 loss:3.100 grdn:106.410 lr:1.0e-05 updt_s:0.299 data_s:0.001
INFO 2025-09-23 15:32:28 ts\train.p

KeyboardInterrupt: 